# Vietnamese OCR Text Correction Pipeline

Corrects OCR/spelling errors in Vietnamese financial-report text using a
fine-tuned ViT5 seq2seq model, while protecting numbers, dates, URLs,
emails, and stock codes from being altered.

**Pipeline:** load model → build/load a Vietnamese + English vocabulary →
for each JSON document, filter which blocks are worth correcting → split
each block into sentences → mask protected spans → correct via the model →
verify the output is safe → restore the protected spans → merge sentences
back → save the corrected JSON, plus QA reports.

## Setup

In [54]:
#protonx
#!pip install -q --force-reinstall transformers==4.38.2 sentencepiece protobuf

## Imports, Paths & Model

In [55]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

INPUT_DIR = "/kaggle/input/datasets/ngkimtrc/pre-bank-2025-tinix"
OUTPUT_DIR = "/kaggle/working/corrected_datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOKENIZER_PATH = "VietAI/vit5-base"
MODEL_PATH= "protonx-models/nano-protonx-legal-tc"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

print("transformers version:")
import transformers
print(transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()
print("Ready")

Device: cuda
transformers version:
4.38.2
Ready


In [56]:
s = "Chù tịch"
ids = tokenizer(s)["input_ids"]
print("tokens :", tokenizer.convert_ids_to_tokens(ids))
print("decoded:", repr(tokenizer.decode(ids, skip_special_tokens=True)))

tokens : ['▁Ch', 'ù', '▁tịch', '</s>']
decoded: 'Chù tịch'


In [57]:
enc = tokenizer("Chù tịch", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=64, num_beams=3)
print("RAW:", repr(tokenizer.decode(out[0], skip_special_tokens=True)))

RAW: 'Chủ tịch'


## Behaviour Configuration

In [58]:
ENABLE_HEADING_CORRECTION = True   # also correct heading blocks, not just paragraphs
MIN_HEADING_WORDS = 1              # skip headings shorter than this
MIN_TEXT_LEN = 3                   # skip text shorter than this (chars)

SIMILARITY_THRESHOLD = 0.80       # reject a correction too different from the original
MIN_LENGTH_RATIO = 0.90            # reject a correction that shrinks the text too much
MAX_LENGTH_RATIO = 1.30           # reject a correction that grows the text too much

MAX_INPUT_LENGTH = 256
MAX_NEW_TOKENS = 64
NUM_BEAMS = 1
DO_SAMPLE = False

DEBUG = False

## Correction Prompt & Protected-Entity Patterns

The prompt instructs the model to fix Vietnamese spelling only. The regex
patterns below identify spans (URLs, emails, dates, numbers, stock codes)
that must never be rewritten — they get masked out before the text reaches
the model (see the masking section further down).

In [59]:
import re
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
DATE_RE = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b")
NUMBER_RE = re.compile(r"[-+]?\d+(?:[.,]\d+)*%?")
#KNOWN_TICKERS = {"A32", "AAA", "VNM", "FPT", "HPG", "VIC", "VCB", "MWG", "CTG", "GAS"}
#STOCK_CODE_RE = re.compile(r"\b(" + "|".join(KNOWN_TICKERS) + r")\b")
SEPARATOR_RE = re.compile(r"^[=\-_*./\\|•·\s]+$")
PAGE_NUMBER_RE = re.compile(r"^\d{1,4}$")
# Dòng chỉ là tick / Có / Không (form công bố)
CHECKMARK_YESNO_RE = re.compile(
    r"^\s*[vV✓√✔]?\s*(Có|Không)\s*$",
    re.IGNORECASE,
)

COMPANY_KEYWORDS = {
    "công ty", "ctcp", "tnhh", "mtv", "tập đoàn", "tổng công ty",
    "ngân hàng", "bộ quốc phòng", "quân khu", "sở giao dịch", "kiểm toán",
    # Big 4 auditors
    "deloitte", "kpmg", "ey", "pwc",
}

In [60]:
# ====================== COMPANY / TICKER ======================
TICKERS_FILE = "/kaggle/input/datasets/ngkimtrc/company-names-2025/tickers.txt"   # hoặc đường dẫn tương ứng trên Kaggle

def load_tickers(path: str) -> set[str]:
    """Load danh sách ticker / tên công ty từ file txt."""
    tickers = set()
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            name = line.strip()
            if not name:
                continue
            # chuẩn hóa về chữ thường để so khớp
            tickers.add(name.lower())
            # cũng thêm bản không dấu nếu cần (tùy chọn)
    return tickers

# Load danh sách ticker/công ty
COMPANY_NAMES = load_tickers(TICKERS_FILE)
print(f"Loaded {len(COMPANY_NAMES):,} company/ticker names")

Loaded 751 company/ticker names


## Vocabulary & Language Detection

Builds (or loads a cached) Vietnamese vocabulary from the Underthesea
dictionary and an English vocabulary from `dwyl/english-words`, then uses
word-overlap ratios to classify a piece of text as Vietnamese, English,
mixed, or unknown. This lets the pipeline skip blocks that are already in
English.

In [61]:
import pickle
import unicodedata
import requests

VOCAB_CACHE_PATH = "vocab_cache.pkl"
ENGLISH_VOCAB_URL = "https://raw.githubusercontent.com/dwyl/english-words/master/words_alpha.txt"
UNDERTHESEA_DICT_DIR = "underthesea_dict"

WORD_RE = re.compile(r"[A-Za-zÀ-ỹ]+")


def normalize_word(word: str) -> str:
    """Unicode-normalize (NFC) and lowercase a word for vocab lookups."""
    return unicodedata.normalize("NFC", word).strip().lower()


def tokenize_words(text: str) -> list[str]:
    """Extract and normalize word tokens from text."""
    return [normalize_word(w) for w in WORD_RE.findall(text)]


def _build_vietnamese_vocab(dict_dir: str) -> set[str]:
    """Build a Vietnamese vocabulary set from the Underthesea dictionary files."""
    vocab = set()
    for root, _, files in os.walk(dict_dir):
        for file in files:
            if file.startswith("."):
                continue
            path = os.path.join(root, file)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        vocab.update(w for w in tokenize_words(line) if len(w) > 1)
            except OSError:
                continue
    return vocab


if os.path.exists(VOCAB_CACHE_PATH):
    print("Loading cached vocabulary...")
    with open(VOCAB_CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    ENGLISH_VOCAB = cache["english"]
    VIETNAMESE_VOCAB = cache["vietnamese"]

else:
    print("Downloading English vocabulary...")
    response = requests.get(ENGLISH_VOCAB_URL)
    response.raise_for_status()
    ENGLISH_VOCAB = {normalize_word(w) for w in response.text.splitlines() if len(w) > 1}
    print(f"English vocabulary: {len(ENGLISH_VOCAB):,} words")

    if not os.path.exists(UNDERTHESEA_DICT_DIR):
        print("Cloning Underthesea dictionary...")
        !git clone --depth 1 https://github.com/undertheseanlp/dictionary.git {UNDERTHESEA_DICT_DIR}

    print("Building Vietnamese vocabulary...")
    VIETNAMESE_VOCAB = _build_vietnamese_vocab(UNDERTHESEA_DICT_DIR)
    print(f"Vietnamese vocabulary: {len(VIETNAMESE_VOCAB):,} words")

    with open(VOCAB_CACHE_PATH, "wb") as f:
        pickle.dump({"english": ENGLISH_VOCAB, "vietnamese": VIETNAMESE_VOCAB}, f)

print(f"English words   : {len(ENGLISH_VOCAB):,}")
print(f"Vietnamese words: {len(VIETNAMESE_VOCAB):,}")

Loading cached vocabulary...
English words   : 370,079
Vietnamese words: 45,586


In [62]:
def detect_language(text: str) -> str:
    """
    Classify `text` as "english", "vietnamese", "mixed", or "unknown"
    based on vocabulary overlap.
    """
    words = tokenize_words(text)
    if not words:
        return "unknown"

    total = len(words)
    en_ratio = sum(1 for w in words if w in ENGLISH_VOCAB) / total
    vi_ratio = sum(1 for w in words if w in VIETNAMESE_VOCAB) / total

    if en_ratio >= 0.80:
        return "english"
    if vi_ratio >= 0.50:
        return "vietnamese"
    return "mixed"

In [63]:
if DEBUG:
    samples = [
        "Since 2001",
        "Independent Auditor's Report",
        "BÁO CÁO TÀI CHÍNH",
        "Cong ty co phan 32",
        "Công ty cổ phần 32",
        "Cash Flow Statement",
        "Thuyết minh báo cáo tài chính",
    ]
    for s in samples:
        print(f"{detect_language(s):12s} | {s}")

## Block & Text Filtering Rules

`should_correct_block` decides which document blocks are worth sending
through the pipeline at all (paragraphs, and optionally long-enough
headings). `should_skip_correction` then decides, per piece of text,
whether it's noise, a URL/email/date/number, a short company name, or
already in English — anything that doesn't need spelling correction.

In [64]:
def is_company_text(text: str) -> bool:
    """True nếu text chứa ticker/tên công ty hoặc keyword tổ chức."""
    normalized = normalize_word(text)
    
    # 1. Khớp với danh sách ticker/tên công ty
    if normalized in COMPANY_NAMES:
        return True
    
    # 2. Khớp keyword chung
    if any(kw in normalized for kw in COMPANY_KEYWORDS):
        return True
    
    return False

def is_noise(text: str) -> bool:
    """True for OCR separators, page numbers, or empty text."""
    stripped = text.strip()
    if not stripped:
        return True
    return bool(PAGE_NUMBER_RE.fullmatch(stripped) or SEPARATOR_RE.fullmatch(stripped))


def should_correct_block(block: dict) -> bool:
    """Decide whether a document block is eligible for spell correction."""
    block_type = block.get("block_type", "").lower()

    if block_type in ("paragraph","list"):
        return True

    if block_type == "heading":
        if not ENABLE_HEADING_CORRECTION:
            return False
        return len(block.get("text", "").split()) >= MIN_HEADING_WORDS

    return False


def should_skip_correction(text: str) -> bool:
    """True if `text` should be left untouched instead of sent to the model."""
    if not text:
        return True

    text = text.strip()

    if len(text) < MIN_TEXT_LEN:
        return True

    if CHECKMARK_YESNO_RE.fullmatch(text):
        if DEBUG:
            print("[SKIP] Yes/No checkbox:", text)
        return True

    if is_noise(text):
        if DEBUG:
            print("[SKIP] Noise")
        return True

    if URL_RE.search(text):
        if DEBUG:
            print("[SKIP] URL")
        return True

    if EMAIL_RE.search(text):
        if DEBUG:
            print("[SKIP] Email")
        return True

    #if STOCK_CODE_RE.fullmatch(text):
        #if DEBUG:
         #   print("[SKIP] Stock code")
        #return True

    if NUMBER_RE.fullmatch(text):
        if DEBUG:
            print("[SKIP] Number")
        return True

    if DATE_RE.fullmatch(text):
        if DEBUG:
            print("[SKIP] Date")
        return True

    if is_company_text(text) and len(text.split()) <= 3:
        if DEBUG:
            print("[SKIP] Company name")
        return True

    if detect_language(text) == "english":
        if DEBUG:
            print("[SKIP] English")
        return True

    return False

## Protecting Entities (Placeholder Masking)

Numbers, dates, URLs, emails, and stock codes are replaced with
`<KEEP_..._####>` placeholders before the text is sent to the model, then
restored afterward. This guarantees the model can never corrupt them,
regardless of what it generates.

In [65]:
# ====================== MASKING + RESTORE (phiên bản ổn định) ======================
import re
from difflib import SequenceMatcher

# Placeholder style đã test: model giữ nguyên tốt
def build_placeholder(tag: str, idx: int) -> str:
    return f"[{tag}_{idx:03d}]"   # [DATE_001], [NUM_002], [URL_001], [EMAIL_001]

# Chỉ mask những thứ thật sự nguy hiểm
DATE_RE = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b")
# Số ≥ 4 chữ số (tránh mask số nhỏ trong ngày)
NUMBER_RE_STRICT = re.compile(r"(?<!\d)[-+]?\d{4,}(?:[.,]\d+)*(?:%?)(?!\d)")
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")

PROTECTED_PATTERNS = [
    ("URL", URL_RE),
    ("EMAIL", EMAIL_RE),
    ("DATE", DATE_RE),
    ("NUM", NUMBER_RE_STRICT),
]

def mask_protected_spans(text: str):
  spans = []
  for tag, pattern in PROTECTED_PATTERNS:
    for m in pattern.finditer(text):
      spans.append({
          'start': m.start(),
          'end': m.end(),
          'text': m.group(),
          'tag': tag,
      })

  if not spans:
    return text, []

  # Ưu tiên span dài hơn khi overlap
  spans.sort(key=lambda s: (s['start'], -(s['end'] - s['start'])))
  filtered, last_end = [], -1
  for s in spans:
    if s['start'] >= last_end:
      filtered.append(s)
      last_end = s['end']

  # 1. Đánh số chỉ mục và lấy originals theo thứ tự Trái -> Phải trước
  originals = [s['text'] for s in filtered]

  # 2. Thay thế chuỗi từ Phải -> Trái để không bị lệch index
  masked = text
  for idx, span in reversed(list(enumerate(filtered, 1))):
    ph = build_placeholder(span['tag'], idx)
    masked = masked[: span['start']] + ph + masked[span['end'] :]

  return masked, originals

def restore_placeholders(text: str, originals: list[str]) -> str:
    if not originals:
        return text

    # Pattern khớp với bất kỳ placeholder nào: [NUM_001], [DATE_002], ...
    ph_pattern = re.compile(r"\[(NUM|DATE|URL|EMAIL)_(\d{3})\]")
    
    restored = text
    matches = list(ph_pattern.finditer(restored))
    
    # Khôi phục từ phải sang trái để không làm xô lệch index chuỗi
    for m in reversed(matches):
        idx = int(m.group(2)) - 1  # Lấy index gốc (0-based)
        if 0 <= idx < len(originals):
            orig_val = originals[idx]
            restored = restored[:m.start()] + orig_val + restored[m.end():]
            
    return restored

## Sentence Splitting

In [66]:
SENTENCE_SPLIT_RE = re.compile(r'(?<=[.!?;:])\s+|\n+')


def split_sentences(text: str) -> list[str]:
    """Split a paragraph into sentences, keeping punctuation attached."""
    if not text or not text.strip():
        return []
    return [s.strip() for s in SENTENCE_SPLIT_RE.split(text.strip()) if s.strip()]


def merge_sentences(sentences: list[str]) -> str:
    """Merge corrected sentences back into a single paragraph."""
    return " ".join(s.strip() for s in sentences if s.strip())

## Correction Core: Sentence → Paragraph

`correct_sentence` runs one sentence end-to-end (mask → generate →
restore → verify) and returns the resulting text plus an optional QA
record — `None` when the sentence was skipped or the model's output
failed verification. `correct_paragraph` splits a block into sentences,
corrects each one, and reassembles them, collecting a QA record for every
sentence that actually went through the model.

In [67]:
def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

def length_ratio(a: str, b: str) -> float:
    return len(b) / max(len(a), 1)

def verify_output(original: str, corrected: str, originals: list[str]) -> bool:
    if not corrected or not corrected.strip():
        return False

    lr = length_ratio(original, corrected)
    sim = similarity(original, corrected)

    # Độ dài
    max_lr = 1.55 if len(original) < 30 else 1.35
    min_lr = 0.55 if len(original) < 30 else 0.75
    if not (min_lr <= lr <= max_lr):
        return False

    # Similarity rất cao → chấp nhận luôn
    if sim >= 0.975:
        return True

    # Mất bất kỳ giá trị được bảo vệ nào → reject ngay
    missing = sum(1 for o in originals if o not in corrected)
    if missing > 0:
        return False
        
    if original.split()[0].lower() in {"v", "✓", "√"} and not corrected.lower().startswith(
        original.split()[0].lower()
    ):
        return False

    return sim >= 0.85

def correct_sentence(sentence: str) -> tuple[str, dict | None]:
    sentence = sentence.strip()

    if should_skip_correction(sentence):
        return sentence, None

    masked, placeholders = mask_protected_spans(sentence)

    inputs = tokenizer(
        masked,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=DO_SAMPLE,
            early_stopping=True,
        )

    corrected = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    if not corrected:
        return sentence, None

    corrected = restore_placeholders(corrected, placeholders)

    if not corrected.strip():
        return sentence, None

    if not verify_output(sentence, corrected, placeholders):
        if DEBUG:
            print(f"\n[Rejected]\n{sentence}\n\n{corrected}\n")
        return sentence, None

    if DEBUG and corrected != sentence:
        print(f"\n[Accepted]\n{sentence}\n\n{corrected}\n")

    qa_record = {
        "original": sentence,
        "corrected": corrected,
        "changed": corrected != sentence,
        "similarity": similarity(sentence, corrected),
        "length_ratio": length_ratio(sentence, corrected),
    }
    return corrected, qa_record

In [68]:
def correct_paragraph(text: str) -> tuple[str, list[dict]]:
    """
    Sửa lỗi đoạn văn bằng cách chia thành từng câu, gom thành 1 BATCH 
    để GPU xử lý song song, sau đó ghép lại thành đoạn văn hoàn chỉnh.
    """
    if not text or not text.strip():
        return text, []

    text = text.strip()
    if should_skip_correction(text):
        return text, []

    sentences = split_sentences(text)
    if not sentences:
        return text, []

    # 1. Thu thập các câu cần sửa và ghi nhận vị trí index
    valid_sentences = []
    valid_indices = []

    for idx, sent in enumerate(sentences):
        sent_str = sent.strip() if isinstance(sent, str) else ""
        if sent_str and not should_skip_correction(sent_str):
            valid_sentences.append(sent_str[:512]) # Giới hạn độ dài câu
            valid_indices.append(idx)

    # Nếu không có câu nào cần sửa, trả về nguyên bản
    if not valid_sentences:
        return text, []

    # 2. XỬ LÝ BATCH: Mã hóa tất cả các câu cùng lúc
    inputs = tokenizer(
        valid_sentences,
        return_tensors="pt",
        padding=True,          # Tự động padding các câu ngắn cho bằng câu dài nhất
        truncation=True,
        max_length=256
    ).to(DEVICE)

    # 3. Đưa toàn bộ Batch vào GPU trong 1 lần duy nhất
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=1,        # Greedy search tối ưu tốc độ
            do_sample=False
        )

    # 4. Giải mã tất cả kết quả trả về
    corrected_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # 5. Ghép kết quả lại đúng vị trí ban đầu & tạo qa_records
    corrected_sentences = list(sentences)
    qa_records = []

    for idx, orig_sent, corrected_sent in zip(valid_indices, valid_sentences, corrected_texts):
        corrected_sentences[idx] = corrected_sent
        
        qa_records.append({
            "original": orig_sent,
            "corrected": corrected_sent,
            "changed": orig_sent != corrected_sent
        })

    # Gộp các câu lại thành paragraph ban đầu
    return merge_sentences(corrected_sentences), qa_records

In [69]:
from difflib import SequenceMatcher
# ============================================================
# DEBUG: xem model thực sự trả gì (chạy TRƯỚC batch processing)
# ============================================================
DEBUG_SAMPLES = [
    "BÁO CÁO LUU CHUYÊN TIÊN TỆ",
    "BÁO CÁO LƯU CHUYÊN TIÊN TỆ",
    "BÀNG CÂN ĐỐI KẾ TOÁN",
    "CÔNG BÓ THÔNG TIN ĐỊNH KỲ",
    "Kế toán trường",
    "Nguyên Thế Anh",
    "Câu trúc doanh nghiệp",
    "THÀNH VIÊN ĐỘC LẬP HĂNG KIỂM TOÁN",
    "Chu tịch Hội đồng quản trị",
    "v Không",
]

for s in DEBUG_SAMPLES:
    masked, ph = mask_protected_spans(s)
    inputs = tokenizer(
        masked,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            num_beams=3,
            do_sample=False,
        )

    raw = tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()
    restored = restore_placeholders(raw, ph)
    ok = verify_output(s, restored, ph)

    print("=" * 60)
    print("IN      :", repr(s))
    print("MASKED  :", repr(masked))
    print("RAW OUT :", repr(raw))
    print("RESTORED:", repr(restored))
    print("VERIFY  :", ok, "| sim=", round(similarity(s, restored), 3))

IN      : 'BÁO CÁO LUU CHUYÊN TIÊN TỆ'
MASKED  : 'BÁO CÁO LUU CHUYÊN TIÊN TỆ'
RAW OUT : 'BÁO CÁO LƯU CHUYÊN TIÊN TỆ'
RESTORED: 'BÁO CÁO LƯU CHUYÊN TIÊN TỆ'
VERIFY  : True | sim= 0.962
IN      : 'BÁO CÁO LƯU CHUYÊN TIÊN TỆ'
MASKED  : 'BÁO CÁO LƯU CHUYÊN TIÊN TỆ'
RAW OUT : 'BÁO CÁO LƯU CHUYÊN TIÊN TỆ'
RESTORED: 'BÁO CÁO LƯU CHUYÊN TIÊN TỆ'
VERIFY  : True | sim= 1.0
IN      : 'BÀNG CÂN ĐỐI KẾ TOÁN'
MASKED  : 'BÀNG CÂN ĐỐI KẾ TOÁN'
RAW OUT : 'BẢNG CÂN ĐỐI KẾ TOÁN'
RESTORED: 'BẢNG CÂN ĐỐI KẾ TOÁN'
VERIFY  : True | sim= 0.95
IN      : 'CÔNG BÓ THÔNG TIN ĐỊNH KỲ'
MASKED  : 'CÔNG BÓ THÔNG TIN ĐỊNH KỲ'
RAW OUT : 'CÔNG BỐ THÔNG TIN ĐỊNH KỲ'
RESTORED: 'CÔNG BỐ THÔNG TIN ĐỊNH KỲ'
VERIFY  : True | sim= 0.96
IN      : 'Kế toán trường'
MASKED  : 'Kế toán trường'
RAW OUT : 'Kế toán trường'
RESTORED: 'Kế toán trường'
VERIFY  : True | sim= 1.0
IN      : 'Nguyên Thế Anh'
MASKED  : 'Nguyên Thế Anh'
RAW OUT : 'Nguyên Thế Anh'
RESTORED: 'Nguyên Thế Anh'
VERIFY  : True | sim= 1.0
IN      : 'Câu trúc doanh ng

In [70]:
print(should_skip_correction("v Không"))
print(bool(CHECKMARK_YESNO_RE.fullmatch("v Không")))

True
True


In [71]:
s = "Chù tịch"
enc = tokenizer(s, return_tensors="pt")
print("input_ids:", enc["input_ids"].tolist())
print("tokens   :", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
print("decoded  :", tokenizer.decode(enc["input_ids"][0], skip_special_tokens=True))

enc = {k: v.to(DEVICE) for k, v in enc.items()}
with torch.no_grad():
    out_ids = model.generate(
        **enc,
        max_new_tokens=64,
        num_beams=3,
        do_sample=False,
    )
print("out_ids  :", out_ids.tolist())
print("raw decode:", repr(tokenizer.decode(out_ids[0], skip_special_tokens=True)))
print("raw +special:", repr(tokenizer.decode(out_ids[0], skip_special_tokens=False)))

input_ids: [[154, 35869, 947, 1]]
tokens   : ['▁Ch', 'ù', '▁tịch', '</s>']
decoded  : Chù tịch
out_ids  : [[0, 1047, 947, 1]]
raw decode: 'Chủ tịch'
raw +special: '<pad> Chủ tịch</s>'


## Batch Processing: Run Over the Dataset

Reads every JSON file in `INPUT_DIR`, corrects the eligible blocks in
place, and writes the corrected document to `OUTPUT_DIR`. QA records from
every sentence that was corrected are collected in `QA_RESULTS` for the
reports below.

In [72]:
import glob
import json
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch
from tqdm.auto import tqdm

BATCH_SIZE = 256
MAX_NEW_TOKENS = 512
NUM_WORKERS = 8

if DEVICE == "cuda":
    model = model.half()

In [73]:
def extract_tasks_from_file(file_path):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        return None, []

    blocks = data.get("blocks", [])
    file_name = Path(file_path).name
    file_tasks = []

    for b_idx, block in enumerate(blocks):
        # -------------------------------------------------------------
        # 1. Trích xuất TEXT trong block (giữ nguyên logic cũ)
        # -------------------------------------------------------------
        if should_correct_block(block):
            text = block.get("text", "")
            if text and not should_skip_correction(text):
                sentences = split_sentences(text)
                for s_idx, sent in enumerate(sentences):
                    sent_str = sent.strip()
                    if sent_str and not should_skip_correction(sent_str):
                        masked, placeholders = mask_protected_spans(sent_str)
                        file_tasks.append({
                            "file_name": file_name,
                            "b_idx": b_idx,
                            "field_type": "text", # Đánh dấu field
                            "s_idx": s_idx,
                            "orig": sent_str,
                            "masked": masked,
                            "placeholders": placeholders,
                        })
            items = block.get("items")
            if isinstance(items, list):
                for i_idx, item in enumerate(items):
                    if not isinstance(item, str):
                        continue
                    item_str = item.strip()
                    if not item_str or should_skip_correction(item_str):
                        continue
                    # item list thường ngắn → không cần split_sentences
                    masked, placeholders = mask_protected_spans(item_str)
                    file_tasks.append({
                        "file_name": file_name,
                        "b_idx": b_idx,
                        "field_type": "items",
                        "s_idx": i_idx,  # vị trí trong list items
                        "orig": item_str,
                        "masked": masked,
                        "placeholders": placeholders,
                    })

        # -------------------------------------------------------------
        # 2. Trích xuất SECTION & SUBSECTION (mới bổ sung)
        # -------------------------------------------------------------
        for field in ["section", "subsection"]:
            val = block.get(field)
            if isinstance(val, str) and val.strip():
                val_str = val.strip()
                if not should_skip_correction(val_str):
                    masked, placeholders = mask_protected_spans(val_str)
                    file_tasks.append({
                        "file_name": file_name,
                        "b_idx": b_idx,
                        "field_type": field, # 'section' hoặc 'subsection'
                        "s_idx": 0,
                        "orig": val_str,
                        "masked": masked,
                        "placeholders": placeholders,
                    })

        # -------------------------------------------------------------
        # 3. Trích xuất HEADING_PATH (mới bổ sung - dạng list)
        # -------------------------------------------------------------
        heading_path = block.get("heading_path")
        if isinstance(heading_path, list):
            for h_idx, h_item in enumerate(heading_path):
                if isinstance(h_item, str) and h_item.strip():
                    h_str = h_item.strip()
                    if not should_skip_correction(h_str):
                        masked, placeholders = mask_protected_spans(h_str)
                        file_tasks.append({
                            "file_name": file_name,
                            "b_idx": b_idx,
                            "field_type": "heading_path",
                            "s_idx": h_idx, # Dùng s_idx để lưu vị trí phần tử trong list
                            "orig": h_str,
                            "masked": masked,
                            "placeholders": placeholders,
                        })

    return (file_path, data), file_tasks

def save_corrected_json(file_name, file_data_map, corrected_results_map, output_dir):
    file_info = file_data_map[file_name]
    data = file_info["data"]
    blocks = data.get("blocks", [])
    changes = corrected_results_map.get(file_name, [])
    
    if changes:
        # Gom nhóm thay đổi theo b_idx và field_type
        # Structure: block_changes[b_idx][field_type][s_idx] = new_text
        block_changes = {}
        for b_idx, field_type, s_idx, new_text in changes:
            if b_idx not in block_changes:
                block_changes[b_idx] = {}
            if field_type not in block_changes[b_idx]:
                block_changes[b_idx][field_type] = {}
            block_changes[b_idx][field_type][s_idx] = new_text

        # Áp dụng các thay đổi vào dữ liệu JSON gốc
        for b_idx, fields_map in block_changes.items():
            if b_idx >= len(blocks):
                continue
            target_block = blocks[b_idx]

            # 1. Cập nhật field 'text'
            if "text" in fields_map:
                orig_text = target_block.get("text", "")
                sentences = split_sentences(orig_text)
                for s_idx, new_sent in fields_map["text"].items():
                    if s_idx < len(sentences):
                        sentences[s_idx] = new_sent
                target_block["text"] = merge_sentences(sentences)

            # 2. Cập nhật 'section'
            if "section" in fields_map:
                target_block["section"] = fields_map["section"][0]

            # 3. Cập nhật 'subsection'
            if "subsection" in fields_map:
                target_block["subsection"] = fields_map["subsection"][0]

            # 4. Cập nhật các phần tử trong 'heading_path'
            if "heading_path" in fields_map:
                h_path = target_block.get("heading_path", [])
                if isinstance(h_path, list):
                    for h_idx, new_h_val in fields_map["heading_path"].items():
                        if h_idx < len(h_path):
                            h_path[h_idx] = new_h_val
                    target_block["heading_path"] = h_path

            # 5. Cập nhật các phần tử trong 'items' (dành cho block_type = "list")
            if "items" in fields_map:
                items = target_block.get("items", [])
                if isinstance(items, list):
                    for i_idx, new_item in fields_map["items"].items():
                        if i_idx < len(items):
                            items[i_idx] = new_item
                    target_block["items"] = items

    out_path = os.path.join(output_dir, file_name)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [74]:
def _reject_reason(original: str, corrected: str, placeholders) -> str:
    if not corrected or not corrected.strip():
        return "empty_output"

    lr = length_ratio(original, corrected)
    max_lr = 1.55 if len(original) < 25 else MAX_LENGTH_RATIO
    min_lr = 0.55 if len(original) < 25 else MIN_LENGTH_RATIO

    if lr < min_lr:
        return f"too_short lr={lr:.3f}"
    if lr > max_lr:
        return f"too_long lr={lr:.3f}"

    sim = similarity(original, corrected)

    # placeholders là list[str]
    missing = any(ph not in corrected for ph in placeholders)

    if sim >= 0.975:
        return "would_accept_high_sim"

    if missing:
        return f"missing_placeholder sim={sim:.3f}"

    if sim < SIMILARITY_THRESHOLD:
        return f"low_sim sim={sim:.3f}"

    return "unknown"

In [75]:
import os
import sys
import glob
import json
import time
import zipfile
import logging
from pathlib import Path
from tqdm.auto import tqdm
import torch

INPUT_DIR = "/kaggle/input/datasets/ngkimtrc/pre-bank-2025-tinix"
OUTPUT_DIR = "/kaggle/working/prebank-corrected_datasets"
ZIP_OUTPUT_DIR = "/kaggle/working/prebankcorrected_datasets_zips"

BATCH_SIZE = 64
FILES_PER_ZIP = 70
START_INDEX = 0
PROGRESS_PATH = os.path.join(OUTPUT_DIR, "progress.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ZIP_OUTPUT_DIR, exist_ok=True)

# Logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s")
logger = logging.getLogger("KagglePipeline")

In [76]:
def load_completed_files() -> set:
    """Lấy danh sách các file đã xử lý từ progress.json và các file zip cũ."""
    completed = set()
    if os.path.exists(PROGRESS_PATH):
        try:
            with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
                completed.update(json.load(f).get("completed", []))
        except Exception as e:
            logger.warning(f"Lỗi đọc progress file: {e}")

    for zip_path in glob.glob(os.path.join(ZIP_OUTPUT_DIR, "*.zip")):
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                for name in zf.namelist():
                    completed.add(name)
                    if name.endswith(".json"):
                        completed.add(name[:-5])
        except zipfile.BadZipFile:
            os.remove(zip_path)
    return completed


def save_progress(completed: set):
    """Lưu tiến trình."""
    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        json.dump({"completed": sorted(list(completed))}, f, ensure_ascii=False, indent=2)


def archive_batch_to_zip(json_filepaths: list):
    """Nén các file JSON thành ZIP và xóa file gốc để tiết kiệm dung lượng Kaggle."""
    if not json_filepaths:
        return
    zip_idx = len(glob.glob(os.path.join(ZIP_OUTPUT_DIR, "corrected_batch_*.zip"))) + 1
    target_zip = os.path.join(ZIP_OUTPUT_DIR, f"corrected_batch_{zip_idx:03d}.zip")

    with zipfile.ZipFile(target_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
        for fp in json_filepaths:
            zf.write(fp, arcname=Path(fp).name)

    for fp in json_filepaths:
        try:
            os.remove(fp)
        except OSError:
            pass
    logger.info(f"Đã đóng gói {len(json_filepaths)} files -> {Path(target_zip).name}")

In [77]:
def infer_batch(batch_tasks: list, model, tokenizer, device: str) -> list:
    """Chạy Model AI trên Batch GPU."""
    batch_masked = [t["masked"] for t in batch_tasks]

    inputs = tokenizer(
        batch_masked,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(device)

    use_amp = (device == "cuda")
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
                do_sample=DO_SAMPLE,
            )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    results = []
    for task, raw_out in zip(batch_tasks, decoded):
        restored = restore_placeholders(raw_out.strip(), task["placeholders"])
        if verify_output(task["orig"], restored, task["placeholders"]):
            results.append(restored)
        else:
            results.append(task["orig"])
    return results

In [78]:
completed = load_completed_files()
all_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.json")))

files_to_process = all_files[max(0, START_INDEX):]
pending_files = [
    f for f in files_to_process 
    if Path(f).name not in completed and Path(f).stem not in completed
]

logger.info(f"Tổng file: {len(all_files)} | Đã làm: {len(completed)} | Còn lại: {len(pending_files)}")

if pending_files:
    start_time = time.time()
    unzipped_queue = []

    for idx, file_path in enumerate(tqdm(pending_files, desc=" Processing Dataset")):
        file_name = Path(file_path).name
        (f_path, data_map), file_tasks = extract_tasks_from_file(file_path)

        corrected_records = []
        if file_tasks and data_map:
            for i in range(0, len(file_tasks), BATCH_SIZE):
                batch_tasks = file_tasks[i : i + BATCH_SIZE]
                batch_results = infer_batch(batch_tasks, model, tokenizer, DEVICE)

                for task, corrected_text in zip(batch_tasks, batch_results):
                    corrected_records.append((task["b_idx"], task["field_type"], task["s_idx"], corrected_text))

            save_corrected_json(
                file_name,
                {file_name: {"data": data_map}},
                {file_name: corrected_records},
                OUTPUT_DIR,
            )

        completed.add(file_name)
        out_json_path = os.path.join(OUTPUT_DIR, file_name)
        if os.path.exists(out_json_path):
            unzipped_queue.append(out_json_path)

        # Gom Batch nén ZIP
        if len(unzipped_queue) >= FILES_PER_ZIP or (idx + 1) == len(pending_files):
            archive_batch_to_zip(unzipped_queue)
            unzipped_queue.clear()
            save_progress(completed)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    logger.info(f"Hoàn thành trong {(time.time() - start_time) / 60:.2f} phút.")

2026-08-12 16:23:00,348 | INFO     | Tổng file: 70 | Đã làm: 0 | Còn lại: 70


 Processing Dataset:   0%|          | 0/70 [00:00<?, ?it/s]

/tmp/ipykernel_260/2081120679.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
2026-08-12 17:24:43,671 | INFO     | Đã đóng gói 70 files -> corrected_batch_001.zip
2026-08-12 17:24:43,712 | INFO     | Hoàn thành trong 61.72 phút.


In [79]:
print(f"Rejected by verify: {len(rejected_logs)}")
print(f"Accepted QA rows  : {len(qa_results)}")

interesting = [
    r for r in rejected_logs
    if r["corrected"] and r["corrected"] != r["original"]
]
interesting.sort(key=lambda x: -x["similarity"])

print("=" * 80)
print(f"Rejected but model CHANGED text: {len(interesting)}")
for r in interesting[:30]:
    print("-" * 60)
    print("FILE :", r["file"])
    print("REASON:", r["reason"])
    print("SIM   :", r["similarity"], "| LR:", r["length_ratio"])
    print("ORIG  :", repr(r["original"][:200]))
    print("OUT   :", repr(r["corrected"][:200]))

import pandas as pd
rej_path = os.path.join(OUTPUT_DIR, "rejected_verify.csv")
pd.DataFrame(rejected_logs).to_csv(rej_path, index=False, encoding="utf-8-sig")
print("Saved:", rej_path)

NameError: name 'rejected_logs' is not defined

## QA Report — Per-Sentence Metrics

Computes similarity, Levenshtein distance, CER, and WER for every
corrected sentence, buckets each into a review category, and saves the
full report as CSV/JSON.

In [ ]:
try:
    import editdistance
except ImportError:
    !pip install -q editdistance
    import editdistance

import os
import pandas as pd


def levenshtein_distance(a: str, b: str) -> int:
    return editdistance.eval(a, b)


def cer(a: str, b: str) -> float:
    if len(a) == 0:
        return 0.0
    return editdistance.eval(a, b) / len(a)


def wer(reference: str, hypothesis: str) -> float:
    ref, hyp = reference.split(), hypothesis.split()
    if len(ref) == 0:
        return 0.0
    return editdistance.eval(ref, hyp) / len(ref)


def rate_review(sim: float) -> str:
    if sim >= 0.995:
        return "Excellent"
    if sim >= 0.985:
        return "Good"
    if sim >= 0.97:
        return "Review"
    return "Manual Check"


rows = []
# Đã đổi QA_RESULTS thành qa_results
for item in qa_results:
    original, corrected = item["original"], item["corrected"]
    sim = similarity(original, corrected)
    rows.append({
        "file": item["file"],
        "block_type": item["block_type"],
        "changed": item["changed"],
        "similarity": round(sim, 5),
        "levenshtein": levenshtein_distance(original, corrected),
        "cer": round(cer(original, corrected), 5),
        "wer": round(wer(original, corrected), 5),
        "review": rate_review(sim),
        "original": original,
        "corrected": corrected,
    })

qa_df = pd.DataFrame(rows)

csv_path = os.path.join(OUTPUT_DIR, "qa_report.csv")
json_path = os.path.join(OUTPUT_DIR, "qa_report.json")
qa_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
qa_df.to_json(json_path, orient="records", force_ascii=False, indent=2)

print("=" * 80)
print("OCR QUALITY REPORT")
print("=" * 80)
print(f"Total blocks         : {len(qa_df):,}")
print(f"Changed              : {qa_df['changed'].sum():,}")
print(f"Average Similarity  : {qa_df['similarity'].mean():.4f}")
print(f"Average CER          : {qa_df['cer'].mean():.5f}")
print(f"Average WER          : {qa_df['wer'].mean():.5f}")
print(f"Average Levenshtein : {qa_df['levenshtein'].mean():.2f}")
print("\nReview Distribution")
print(qa_df["review"].value_counts())

print("\n" + "=" * 80)
print("Top 20 Most Changed Blocks")
display(qa_df.sort_values(by="cer", ascending=False).head(20))

print(f"\nCSV  : {csv_path}")
print(f"JSON : {json_path}")
print("=" * 80)

## QA Report — Per-File Summary

Aggregates the per-sentence QA report by file so problem files (highest
average CER, most corrections, most items flagged for manual review) are
easy to spot.

In [ ]:
if len(qa_df) == 0:
    print("QA report is empty.")
else:
    summary = (
        qa_df.groupby("file")
        .agg(
            total_blocks=("file", "count"),
            corrected_blocks=("changed", "sum"),
            avg_similarity=("similarity", "mean"),
            avg_cer=("cer", "mean"),
            avg_wer=("wer", "mean"),
            avg_levenshtein=("levenshtein", "mean"),
            worst_cer=("cer", "max"),
            manual_review=("review", lambda x: (x == "Manual Check").sum()),
        )
        .reset_index()
    )
    summary["change_rate"] = summary["corrected_blocks"] / summary["total_blocks"]
    summary = summary[[
        "file", "total_blocks", "corrected_blocks", "change_rate",
        "avg_similarity", "avg_cer", "avg_wer", "avg_levenshtein",
        "worst_cer", "manual_review",
    ]]
    summary = summary.sort_values(by=["avg_cer", "manual_review"], ascending=False)

    summary_csv = os.path.join(OUTPUT_DIR, "file_summary.csv")
    summary_json = os.path.join(OUTPUT_DIR, "file_summary.json")
    summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    summary.to_json(summary_json, orient="records", indent=2, force_ascii=False)

    print("=" * 80)
    print("FILE LEVEL SUMMARY")
    print("=" * 80)
    print(f"\nTotal Files : {len(summary):,}\n")

    print("Worst 20 Files (Highest Average CER)")
    display(summary.head(20))

    print("\n" + "=" * 80)
    print("Top 20 Files With Most Corrections")
    display(summary.sort_values("corrected_blocks", ascending=False).head(20))

    print("\n" + "=" * 80)
    print("Top 20 Files Needing Manual Review")
    display(summary.sort_values("manual_review", ascending=False).head(20))

    print(f"\nCSV  : {summary_csv}")
    print(f"JSON : {summary_json}")
    print("=" * 80)

In [ ]:
test_cases = [
    # --- Heading báo cáo ---
    ("BÁO CÁO LUU CHUYÊN TIÊN TỆ", "BÁO CÁO LƯU CHUYỂN TIỀN TỆ"),
    ("BÁO CÁO LƯU CHUYÊN TIÊN TỆ", "BÁO CÁO LƯU CHUYỂN TIỀN TỆ"),
    ("BÁO CÁO KẾT QUÀ HOẠT ĐỘNG KINH DOANH", "BÁO CÁO KẾT QUẢ HOẠT ĐỘNG KINH DOANH"),
    ("BÀNG CÂN ĐỐI KẾ TOÁN", "BẢNG CÂN ĐỐI KẾ TOÁN"),
    ("CÔNG BÓ THÔNG TIN ĐỊNH KỲ", "CÔNG BỐ THÔNG TIN ĐỊNH KỲ"),
    ("THÀNH VIÊN ĐỘC LẬP HĂNG KIỂM TOÁN", "THÀNH VIÊN ĐỘC LẬP HÃNG KIỂM TOÁN"),
    ("BẢN THUYẾT MINH BÁO CÁO TÀI CHÍNH", "BẢN THUYẾT MINH BÁO CÁO TÀI CHÍNH"),  # đã đúng
    ("BÁO CÁO CỦA BAN ĐIỀU HÀNH", "BÁO CÁO CỦA BAN ĐIỀU HÀNH"),

    # --- Chức danh / tên ---
    ("Kế toán trường", "Kế toán trưởng"),
    ("Nguyên Thế Anh", "Nguyễn Thế Anh"),
    ("Chu tịch Hội đồng quản trị", "Chủ tịch Hội đồng quản trị"),
    ("Tổng Giám đôc", "Tổng Giám đốc"),
    ("Phó Tổng giám đôc", "Phó Tổng giám đốc"),

    # --- Câu nội dung ngắn ---
    ("Cho năm tài chính kết thúc ngày 31/12/2025", "Cho năm tài chính kết thúc ngày 31/12/2025"),
    ("đính kèm báo cáo tài chính", "đính kèm báo cáo tài chính"),
    ("Câu trúc doanh nghiệp", "Cấu trúc doanh nghiệp"),
    ("Lợi nhuận sau thuế chưa phân phôi", "Lợi nhuận sau thuế chưa phân phối"),
    ("Doanh thu bán hàng và cung cấp dịch vụ", "Doanh thu bán hàng và cung cấp dịch vụ"),

    # --- Không được sửa (bảo vệ) ---
    ("v Không", "v Không"),
    ("Có", "Có"),
    ("31/12/2025", "31/12/2025"),
    ("A32", "A32"),
]

def run_tests(fn):
    """fn(text) -> corrected text"""
    ok = 0
    for orig, expected in test_cases:
        out = fn(orig)
        status = "OK" if out == expected else "FAIL"
        if status == "OK":
            ok += 1
        print(f"{status:4} | {orig} → {out}")
        if status == "FAIL":
            print(f"     expected: {expected}")
    print(f"\n{ok}/{len(test_cases)} PASS")

In [ ]:
import gc, torch

# xóa biến model/tokenizer cũ nếu còn
for name in ["model", "mdl", "tokenizer", "tok"]:
    if name in dir():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free/1024**3:.2f} / {total/1024**3:.2f} GiB")

In [ ]:
import os, gc, torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TESTS = [
    "BÁO CÁO LUU CHUYÊN TIÊN TỆ",
    "BÁO CÁO LƯU CHUYÊN TIÊN TỆ",
    "BÀNG CÂN ĐỐI KẾ TOÁN",
    "CÔNG BÓ THÔNG TIN ĐỊNH KỲ",
    "Kế toán trường",
    "Nguyên Thế Anh",
    "Câu trúc doanh nghiệp",
    "THÀNH VIÊN ĐỘC LẬP HĂNG KIỂM TOÁN",
    "Chu tịch Hội đồng quản trị",
    "v Không",
]

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM free: {free/1024**3:.2f}/{total/1024**3:.2f} GiB")

def test_one(name: str):
    clear_gpu()
    print("=" * 80)
    print("MODEL:", name)
    try:
        tok = AutoTokenizer.from_pretrained(name, use_fast=False)
        mdl = AutoModelForSeq2SeqLM.from_pretrained(
            name,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
            low_cpu_mem_usage=True,
        ).to(DEVICE)
        mdl.eval()
    except Exception as e:
        print("LOAD FAIL:", type(e).__name__, str(e)[:250])
        clear_gpu()
        return

    for t in TESTS:
        try:
            enc = tok(t, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
            with torch.no_grad():
                out = mdl.generate(**enc, max_new_tokens=64, num_beams=3)
            print(f"  {t}\n    → {tok.decode(out[0], skip_special_tokens=True).strip()}")
        except Exception as e:
            print(f"  {t}\n    → ERROR {e}")

    del tok, mdl
    clear_gpu()
    print("Done.\n")

In [ ]:
# 3) Diacritic ViT5 base (~900MB)
test_one("nrl-ai/vn-diacritic-vit5-base")

In [ ]:
# 5) BARTpho correction (~1.5GB) — restart kernel nếu OOM
test_one("bmd1905/vietnamese-correction-v2")

In [ ]:
# 4) Spell correction base (~900MB) — chỉ khi VRAM trống > 2.5GiB
test_one("nrl-ai/vn-spell-correction-base")